## Classify Romansh vs. non-Romansh Text
This script illustrates how we can use the romansh_lemmatizer to distinguish between Romansh and non-Romansh text.

### Preliminaries: Import

In [3]:
# 1. Imports and initialisation
from romansh_lemmatizer import Lemmatizer

### Analyzing the Text
Define the example texts, then initialise the lemmatizer (non idiom-specific because we want to classify the text) and define a function that helps us classify the text.

The function computes an average coverage score by averaging the individual idiom scores. It also computes a global coverage score by counting how many tokens are in *any* Romansh variety.

In [7]:
# 2. Define example texts
samples = {
    "Romansh (Surmiran)": "La mamma fò pang e la feglia ligia an tgesa.",
    "Romansh (Puter)": "La mamma fo paun e la figlia legia in chesa.",
    "German": "Die Mutter backt Brot und die Tochter liest im Haus.",
    "French": "La mère fait du pain et la fille lit dans la maison.",
    "Italian": "La madre fa il pane e la figlia legge in casa.",
}

In [31]:
# 3. Function to classify text
lemmatizer = Lemmatizer()

def classify_romansh_vs_nonromansh(text):
    doc = lemmatizer(text)

    # Get all idiom scores (coverage ratios)
    idiom_scores = doc.idiom_scores
    # Compute Romansh coverage: mean or sum over all idioms
    romansh_score = sum(idiom_scores.values()) / len(idiom_scores)

    # For absolute coverage, we can also count how many tokens matched any idiom
    total_tokens = len(doc.tokens)
    romansh_hits = sum(1 for tok in doc.tokens if any(tok.lemmas))
    coverage = romansh_hits / total_tokens if total_tokens else 0.0

    # Classify as Romansh if enough tokens are recognized
    # coverage: in any idiom, romansh_score: average
    label = "Romansh" if (coverage > 0.7 and  romansh_score > 0.5) else "Non-Romansh"

    # Get the idiom with the highest individual score
    max_score = max(idiom_scores.values())
    best_items = [(k, v) for k, v in idiom_scores.items() if v == max_score]


    max_score = max(idiom_scores.values())
    best_items = [(k, v) for k, v in idiom_scores.items() if v == max_score]

    return {
        "avg_coverage": romansh_score,
        "global_coverage": coverage,
        "label": label,
        "best_idioms": [idiom.value for idiom, _ in best_items],
        "best_idiom_scores": [score for _, score in best_items],
    }


### Run Classification and Pretty Print Results
Here, we run the previously defined function and print the results in a nice format.

In [33]:
# Run classification
print("Romansh vs Non-Romansh classification\n")

for name, text in samples.items():
    result = classify_romansh_vs_nonromansh(text)

    print(f"Text: '{text}' in {name}")
    print(f"  → Assigned Label: {result['label']}")
    # Handle multiple best idioms
    idioms = result["best_idioms"]
    scores = result["best_idiom_scores"]

    if len(idioms) == 1:
        print(f"  → clostest idiom: {idioms[0]} ({scores[0]:.3f})")
    else:
        print("  → closest idioms:")
        for idiom, score in zip(idioms, scores):
            print(f"     - {idiom} ({score:.3f})")

    print(f"  → Romansh coverage (any idiom): {result['global_coverage']:.3f}")
    print(f"  → Averaged per-idiom Romansh coverage: {result['avg_coverage']:.3f}\n")



Romansh vs Non-Romansh classification

Text: 'La mamma fò pang e la feglia ligia an tgesa.' in Romansh (Surmiran)
  → Assigned Label: Romansh
  → clostest idiom: rm-surmiran (0.909)
  → Romansh coverage (any idiom): 0.909
  → Averaged per-idiom Romansh coverage: 0.576

Text: 'La mamma fo paun e la figlia legia in chesa.' in Romansh (Puter)
  → Assigned Label: Romansh
  → clostest idiom: rm-puter (0.909)
  → Romansh coverage (any idiom): 0.909
  → Averaged per-idiom Romansh coverage: 0.682

Text: 'Die Mutter backt Brot und die Tochter liest im Haus.' in German
  → Assigned Label: Non-Romansh
  → closest idioms:
     - rm-sursilv (0.273)
     - rm-sutsilv (0.273)
     - rm-puter (0.273)
     - rm-vallader (0.273)
  → Romansh coverage (any idiom): 0.273
  → Averaged per-idiom Romansh coverage: 0.227

Text: 'La mère fait du pain et la fille lit dans la maison.' in French
  → Assigned Label: Non-Romansh
  → clostest idiom: rm-puter (0.538)
  → Romansh coverage (any idiom): 0.538
  → Average